## Feature Engineering and Text Represenation
This notebook transforms the pre-processed text to generate different types of features representations for the clothing reviews. Note, both 'Title' and 'Review Text' will be considered. This project will implement, Bag-Of-Words, TF-IDF, Glove and BERT embedding techniques.

In [2]:
#import libraries

import pandas as pd
import numpy as np
#import re
#import string             
#import nltk
from collections import Counter
#from nltk.corpus import stopwords
#from nltk.stem import WordNetLemmatizer



#nltk.download('punkt')
#nltk.download('stopwords')
#nltk.download('wordnet')  #consider nltk for lemmatize?

In [3]:
import ast

file_path = "../data/cleaned_df.csv"

try:
    df = pd.read_csv(
        file_path,
        dtype={
        "Unnamed: 0": int,
        "Clothing ID": int,
        "Age": int,
        "Title": str,
        "Review Text": str,
        "Rating": int,
        "Recommended IND": int,
        "Positive Feedback Count": int,
        "Division Name": str,
        "Department Name": str,
        "Class Name": str,
        "Length": int,
        },
        converters={
        "r_tokens": ast.literal_eval,  # Parse string back to list
        "t_tokens": ast.literal_eval,   # "                       "
        }
    )
    
    print("It's loaded big dawg")
except FileNotFoundError:
    print("It didn't work big dawg")

It's loaded big dawg


**Bag of Words**

Count vector representation

In [4]:
#manual count vector representation

#read vocab to dict
man_vmap = {}
map_path = '../data/vmap.txt'
with open(map_path, 'r',) as file:
    for line in file:
        k,v = line.split(':')
        man_vmap[k] = v

#combined tokens vectorisation
df['combined_tokens'] = df['t_tokens'] + df['r_tokens']
c_t = df['combined_tokens']
#sparse count vector representation (vmap_index:freq)
mcnt_vct = []

for tokens in c_t:
    index = [int(man_vmap[tok]) for tok in tokens if tok in man_vmap]
    freq_cnt = Counter(index)
    sparse = [f'{idx}:{freq}' for idx,freq in freq_cnt.items()]
    sparse_vstr = ','.join(sparse)
    mcnt_vct.append(sparse_vstr)


#write to txt file
#i = df['rev_idx']
mcnt_vct_path = '../data/count_vectors.txt'
with open(mcnt_vct_path, 'w') as f_out:
    for rev_idx, vector_str in zip(df['rev_idx'], mcnt_vct):
        f_out.write(f"#{rev_idx},{vector_str}\n")

In [5]:
# rbg 100 - # 599 non-printable control bytes

In [8]:
#Glove embedding, TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

glove_file = '../glove.6B/glove.6B.100d.txt' 
glove_model = {}
# read all word embeddings from GloVe and store each word with vector in dictionary
with open(glove_file, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split()
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove_model[word] = vector

print(f"Loaded {len(glove_model)} GloVe word vectors") 

# prepare tf-idf vectorizer on combined tokens
def identity_preprocessor(x):
    return x

def identity_tokenizer(x):
    return x

tfidf_vectorizer = TfidfVectorizer(
    tokenizer=identity_tokenizer,
    preprocessor=identity_preprocessor,
    token_pattern=None
)

tfidf_vectorizer.fit(df['combined_tokens'])
tfidf_scores = dict(zip(tfidf_vectorizer.get_feature_names_out(), tfidf_vectorizer.idf_))

# empty lists to store representations
embedding_unweighted = []
embedding_weighted = []

# loop over each tokenized review
for review in df['combined_tokens']:
    vectors = []
    weights = []
    
    for word in review:
        if word in glove_model:
            vectors.append(glove_model[word])
            # weighted: use TF-IDF score if available
            weights.append(tfidf_scores.get(word, 1.0))

    #if review has known words
    if vectors:
        # unweighted: simple average of embeddings
        embedding_unweighted.append(np.mean(vectors, axis=0))
        # weighted: TF-IDF weighted average
        weighted_vec = np.average(vectors, axis=0, weights=weights)
        embedding_weighted.append(weighted_vec)

    # if no known words, zero vector    
    else:
        embedding_unweighted.append(np.zeros(len(next(iter(glove_model.values())))))
        embedding_weighted.append(np.zeros(len(next(iter(glove_model.values())))))

print("Generated embedding-based unweighted and TF-IDF weighted feature vectors using GloVe")

Loaded 400000 GloVe word vectors
Generated embedding-based unweighted and TF-IDF weighted feature vectors using GloVe


In [10]:
import pickle

df.to_pickle("../data/df_with_tokens.pkl") 
print("Saved DataFrame with combined_tokens as data/df_with_tokens.pkl") 

# Save the GloVe lookup dictionary 
with open("../models/glove_model.pkl", "wb") as f: 
    pickle.dump(glove_model, f) 
print("Saved GloVe word embedding dictionary as models/glove_model.pkl") 

# Save TF-IDF vectorizer model 
with open("../models/tfidf_vectorizer.pkl", "wb") as f: 
    pickle.dump(tfidf_vectorizer, f) 
print("Saved TF-IDF vectorizer as models/tfidf_vectorizer.pkl") 
    
# Save numerical embeddings (NumPy) 
np.save("../features/embedding_unweighted.npy", np.array(embedding_unweighted))
np.save("../features/embedding_weighted.npy", np.array(embedding_weighted)) 
print("Saved NumPy embeddings as features/*.npy") 

# Save text versions (readable) 

with open("../features/embedding_vectors.txt", "w", encoding="utf-8") as f: 
    for vec in embedding_unweighted: 
        f.write(",".join(str(x) for x in vec) + "\n") 
        
with open("../features/tfidf_embedding_vectors.txt", "w", encoding="utf-8") as f: 
    for vec in embedding_weighted: 
        f.write(",".join(str(x) for x in vec) + "\n") 

print("Saved text embeddings as features/*.txt") 

print("\n All models, data, and embeddings saved successfully!")


Saved DataFrame with combined_tokens as data/df_with_tokens.pkl
Saved GloVe word embedding dictionary as models/glove_model.pkl
Saved TF-IDF vectorizer as models/tfidf_vectorizer.pkl
Saved NumPy embeddings as features/*.npy
Saved text embeddings as features/*.txt

 All models, data, and embeddings saved successfully!
